In [2]:
from pathlib import Path
import pandas as pd
from cyvcf2 import VCF
from tqdm import tqdm
import json

In [ ]:
snp_only_dir = Path("./snp_only")

vcf_files = sorted(snp_only_dir.glob("*.vcf"))

In [ ]:
# Collect all unique SNP sites across samples 
all_sites = set()
sample_sites = {}

print("🔍 Reading SNP sites from all VCFs...")
for vcf_file in tqdm(vcf_files):
    sample_name = vcf_file.stem.replace("_snps_only", "")
    reader = VCF(str(vcf_file))
    sites = []
    for record in reader:
        # Use (CHROM, POS, REF, ALT) tuple for uniqueness
        for alt in record.ALT:
            site = (record.CHROM, record.POS, record.REF, str(alt))
            sites.append(site)
            all_sites.add(site)
    sample_sites[sample_name] = sites

In [ ]:
all_sites = sorted(all_sites)

print(f"✅ Total unique SNP sites: {len(all_sites)}")

In [ ]:
matrix = []
for sample_name, sites in tqdm(sample_sites.items(), desc="🧬 Building SNP matrix"):
    sites_set = set(sites)
    row = [1 if site in sites_set else 0 for site in all_sites]
    matrix.append(row)

In [ ]:
# siguro next time, paki-add yung country and all huhu kung need
columns = [f"{chrom}_{pos}_{ref}_{alt}" for chrom, pos, ref, alt in all_sites]
df = pd.DataFrame(matrix, columns=columns, index=sample_sites.keys())

print("✅ SNP Matrix shape:", df.shape)

In [ ]:
# display DataFrame
display(df)

In [ ]:
df.to_csv("snp_matrix.csv")
print("📁 Saved SNP matrix to snp_matrix.csv")